In [1]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
import operator

/Users/sanjaymahto/About-AI/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
llm_model = "llama-3.1-8b-instant"
llm = ChatGroq(
    model=llm_model,
    temperature=0,
)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x1170c2bb0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1170c9190>, model_name='llama-3.1-8b-instant', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
class EvaluationSchema(BaseModel): 
    feedback: str = Field(..., description="Detailed feedback on the quality of the essay.")
    score: int = Field(description='Score out of 10', ge=0, le=10)

In [16]:
structure_model = llm.with_structured_output(EvaluationSchema)

In [ ]:
essay = """"""

In [30]:
prompt = f""" 
         Evaluate the language quality of the following essay and provide feedback and assign a score out of 10 \n {essay}
"""

result = structure_model.invoke(prompt)

BadRequestError: Error code: 400 - {'error': {'message': "tool call validation failed: parameters for tool EvaluationSchema did not match schema: errors: [missing properties: 'score']", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=EvaluationSchema> {"feedback": "The essay provides a comprehensive analysis of India\'s position in the age of AI, highlighting both the opportunities and challenges. The writer effectively uses examples to illustrate the potential applications of AI in various sectors, such as agriculture, healthcare, and education. However, the essay could benefit from a more nuanced discussion of the digital divide and job displacement concerns. Additionally, the writer could have provided more concrete policy recommendations to address these issues. Overall, the essay demonstrates a good understanding of the topic and presents a clear argument. Score: 8/10"} </function>'}}

In [19]:
result

EvaluationSchema(feedback="The essay is well-structured and provides a comprehensive overview of Rohit Sharma's career. The language is clear and concise, and the writer has done a good job of highlighting Rohit's achievements and records. However, the essay could benefit from more analysis and critique of Rohit's batting style and approach. Additionally, some of the sentences are quite long and could be broken up for better readability. Overall, the essay is well-written and provides a good introduction to Rohit Sharma's career.", score=8)

In [20]:
result.feedback


"The essay is well-structured and provides a comprehensive overview of Rohit Sharma's career. The language is clear and concise, and the writer has done a good job of highlighting Rohit's achievements and records. However, the essay could benefit from more analysis and critique of Rohit's batting style and approach. Additionally, some of the sentences are quite long and could be broken up for better readability. Overall, the essay is well-written and provides a good introduction to Rohit Sharma's career."

In [21]:
result.score

8

In [7]:
class UPSEState(TypedDict): 
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    inidividual_scores: Annotated[list[int], operator.add]
    avg_score: float

In [8]:
graph = StateGraph(UPSEState)

In [27]:
def evaluate_language(state: UPSEState) -> UPSEState: 
    prompt = f"Evaluate the language quality of the following essay and provide feedback and assign a score out of 10 \n {state['essay']}"
    result = structure_model.invoke(prompt)
    
    return {
        'language_feedback': result.feedback, 
        'inidividual_scores': [result.score]
    }

def evaluate_analysis(state: UPSEState) -> UPSEState: 
    prompt = f"Evaluate the depth of the analysis of the following essay and provide feedback and assign a score out of 10 \n {state['essay']}"
    result = structure_model.invoke(prompt)
    
    return {
        'analysis_feedback': result.feedback, 
        'inidividual_scores': [result.score]
    }


def evaluate_thought(state: UPSEState) -> UPSEState: 
    prompt = f"Evaluate the clarity of thought of the following essay and provide feedback and assign a score out of 10 \n {state['essay']}"
    result = structure_model.invoke(prompt)
    
    return {
        'clarity_feedback': result.feedback, 
        'inidividual_scores': [result.score]
    } 



def final_evaluation(state: UPSEState) -> UPSEState: 
    prompt = f"Based on the following feedback create summarized feedback \n Language Feedback: {state['language_feedback']} \n Depth of Analysis Feedback: {state['analysis_feedback']} \n Clarity of Thought Feedback: {state['clarity_feedback']}"
    overall_feedback = llm.invoke(prompt).content
    
    # total = 0 
    # for i in state['inidividual_scores']: 
    #     total += i
        
    # avg_score = total / len(state['inidividual_scores'])
    
    avg_score = sum(state['inidividual_scores']) / len(state['inidividual_scores'])
    
    return {
        'overall_feedback': overall_feedback, 
        'avg_score': avg_score
    } 

In [10]:
# Nodes

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

# Add edges 
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()


In [25]:
print(essay)

India in the Age of AI
As the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.

India's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for All) with a fo

In [28]:
inital_state = {
    'essay': essay,
    'inidividual_scores': []
}


workflow.invoke(inital_state)

BadRequestError: Error code: 400 - {'error': {'message': "tool call validation failed: parameters for tool EvaluationSchema did not match schema: errors: [missing properties: 'score']", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=EvaluationSchema> {"feedback": "The essay provides a comprehensive analysis of India\'s position in the age of AI, highlighting both the opportunities and challenges. The author effectively uses examples to illustrate the potential applications of AI in various sectors, such as agriculture, healthcare, and education. However, the essay could benefit from a more nuanced discussion of the digital divide and job displacement concerns. Additionally, the author could have provided more concrete policy recommendations to address these issues. Overall, the essay demonstrates a good understanding of the topic and presents a clear argument, but could be strengthened with more depth and specificity in certain areas. Score: